In [1]:
# === Crudos -> *_clean.csv respetando encabezados 'nm,A' ===
from pathlib import Path
import io, re
import numpy as np
import pandas as pd

# Estás parado en la carpeta que contiene las subcarpetas (AMhMg_7_5g, AMhMg_15g, ..., ALB_AMhMg)
ROOT = Path(".")
PATTERN = "**/*.csv"             # recursivo
NA_TOKENS = {"XXX.XXX","XXX","xx.x","—","--","","NaN","nan"}
OVERWRITE = True                  # True: reescribe si ya existe el _clean

def detect_sep_and_decimal(text: str):
    lines = [ln for ln in text.splitlines() if ln.strip()][:60]
    blob = "\n".join(lines)
    counts = {",": blob.count(","), ";": blob.count(";"), "\t": blob.count("\t")}
    sep = max(counts, key=counts.get) if max(counts.values())>0 else ","
    # ¿coma decimal?
    comma_dec = len(re.findall(r"\d+,\d+", blob))
    dot_dec   = len(re.findall(r"\d+\.\d+", blob))
    decimal = "," if (comma_dec > dot_dec and sep != ",") else "."
    return sep, decimal

def read_two_cols_as_nm_A(p: Path) -> pd.DataFrame:
    """
    Lee cualquier CSV (con o sin encabezado) y devuelve un DataFrame con columnas EXACTAS 'nm','A'.
    - Autodetecta separador y decimal
    - Convierte tokens raros a NaN
    - Fuerza a numérico y descarta filas no numéricas
    """
    txt = p.read_text(errors="ignore")
    sep, dec = detect_sep_and_decimal(txt)

    # Estrategia robusta: leer SIN encabezado y poner nombres fijos ['nm','A'].
    # Si el archivo trae más columnas, tomamos sólo las 2 primeras.
    df = pd.read_csv(
        io.StringIO(txt),
        sep=sep, engine="python",
        header=None,               # ignoramos encabezados del proveedor
        usecols=[0,1],             # primeras 2 columnas
        names=["nm","A"],          # estandarizamos SIEMPRE
        na_values=list(NA_TOKENS),
        decimal=dec,
        comment="#",
        dtype=str,                 # primero como string (para limpiar)
    )

    # Coerción a numérico y limpieza
    df["nm"] = pd.to_numeric(df["nm"].str.strip(), errors="coerce")
    df["A"]  = pd.to_numeric(df["A"].str.strip(),  errors="coerce")
    df = (df.dropna(subset=["nm","A"])
            .drop_duplicates()
            .sort_values("nm")
            .reset_index(drop=True))
    return df

manifest = []
for f in sorted(ROOT.glob(PATTERN)):
    if not f.is_file():
        continue
    if f.stem.endswith("_clean"):
        continue
    out = f.with_name(f.stem + "_clean.csv")
    if out.exists() and not OVERWRITE:
        manifest.append({"raw": str(f), "clean": str(out), "status": "skip (exists)"})
        continue
    try:
        df = read_two_cols_as_nm_A(f)
        df.to_csv(out, index=False)  # <-- encabezados EXACTOS: nm,A
        status = "ok" if len(df) else "empty-after-clean"
    except Exception as e:
        status = f"error: {e}"
    manifest.append({"raw": str(f), "clean": str(out), "rows": len(df) if 'df' in locals() else 0, "status": status})

pd.DataFrame(manifest)

,raw,clean,rows,status
0,ALB_MhMg/ALB_MhMg.csv,ALB_MhMg/ALB_MhMg_clean.csv,405,ok
1,AMhMg_15g/AMhMg_15g.csv,AMhMg_15g/AMhMg_15g_clean.csv,374,ok
2,AMhMg_18g/AMhMg_18g.csv,AMhMg_18g/AMhMg_18g_clean.csv,376,ok
3,AMhMg_22_5g/AMhMg_22_5g.csv,AMhMg_22_5g/AMhMg_22_5g_clean.csv,381,ok
4,AMhMg_7_5g/AMhMg_7_5g.csv,AMhMg_7_5g/AMhMg_7_5g_clean.csv,405,ok
